# NanoCore-S1 Training Notebook

Auto-runs Stage 1 base training on Kaggle/Colab GPU, commits results back to GitHub.

This notebook eliminates GPU waste by **validating the training pipeline locally first** via `--dry-run`, then launching the full cloud training.

## Step 1: Verify Local Dry Run ✅

Before touching cloud GPU, we verify the training pipeline works on your local RTX 2070:

In [ ]:
# Install dependencies (Colab/Kaggle runtime)
!pip install torch transformers datasets 2>&1 | tail -3

In [ ]:
# Clone and set up the repo
%cd /content  # Use /kaggle/working on Kaggle
!git clone https://huggingface.co/spaces/placeholder 2>/dev/null || true  # Placeholder
# For now, just use git to get the repo
!git clone https://github.com/yourname/nanocore-s1.git 2>/dev/null || git clone https://github.com/ikaaros/nanocore-s1.git
%cd nanocore-s1
!pip install -e . 2>/dev/null || true

In [ ]:
# Run dry-run to verify pipeline works
%time python scripts/train_base.py --dry-run

## Step 2: Full Training on Cloud GPU

Once dry-run passes, run full pretraining. Uses FineWeb-EDU (free from HuggingFace).  
Estimated time: 15-20 min on T4/A10, 8-10 min on T4/V100.

In [ ]:
# Full Stage 1 training (200M tokens)
# batch_size=32, grad_accum=16, effective_batch=512
%time python scripts/train_base.py --batch-size=32 --grad-accum=16 --max-steps=1000

## Step 3: Auto-Commit Results to GitHub

Automatically commits the trained checkpoint and report back to GitHub.

In [ ]:
# Set up git for auto-commit
!git config --global user.email "nanocore@bot.com"
!git config --global user.name "NanoCore Bot"

# Use GitHub token for push (set as secret in Colab/Kaggle)
import os
token = os.environ.get('GH_TOKEN') or os.environ.get('GITHUB_TOKEN')
if token:
    remote = f'https://{token}@github.com/ikaaros/nanocore-s1.git'
    !git remote set-url origin {remote}
else:
    print('Set GH_TOKEN env var for auto-push')

In [ ]:
# Commit and push results
import json, os

# Check the training report
if os.path.exists('models/training_report.json'):
    with open('models/training_report.json') as f:
        report = json.load(f)
    print(json.dumps(report, indent=2))

# Commit
!git add models/nanocore-s1-base.pt models/training_report.json
!git commit -m "train: Stage 1 base model (auto from cloud GPU)" || true
!git push origin main 2>/dev/null || print('No token set - push skipped')

## Step 4: Download Results Locally

If you prefer manual download instead of auto-push:

In [ ]:
from google.colab import files  # Colab only
files.download('models/nanocore-s1-base.pt')